In [7]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re
from itertools import batched

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))

In [8]:
id_mapping_file = Path(project_root / "data/people/prepping4loading/people_id_mapping.json")
matched_file = Path(project_root / "data/people/prepping4loading/people_matched.json")

with open(id_mapping_file, "r") as f:
    mapping_dict = json.load(f)

with open(matched_file, "r") as f:
    matched = json.load(f)

# rprint(dict(list(matched.items())[:2]))

# check if there could be collisions if silently overwriting entries
# check_matched_unified_ids = [v["unified_id"] for v in matched.values()]
# check_matched_old_ids = [v["person_id"] for v in matched.values()]
# rprint(f"total matched: {len(matched)}, total old matched ids: {len(check_matched_old_ids)}")

matched_dict = {
    v["person_id"]: {
        "unified_id": v["unified_id"],
        "entries": v["occurrences"]
    }
    for k, v in matched.items()
}
# rprint(dict(list(matched_dict.items())[:1]))

mapping_id_dict = {v["person_id_old"]: v for k, v in mapping_dict.items()}
# rprint(dict(list(mapping_id_dict.items())[:2]))


b2p_matched_list = []
b2p_matched_data = {}
seen_combinations = set()
update_count = 0
problems = {}

roles = [
    "is_author",
    "is_editor",
    "is_contributor",
    "is_translator"
]
updates = {}
problem_count = 0

for id_old, person in matched_dict.items():
# for id_old, person in list(matched_dict.items())[:5]:
    if not id_old in mapping_id_dict:
        problem_count += 1
        problems[id_old] = person
        continue

    data = mapping_id_dict[id_old]
    person_id = data["person_id"]
    unified_id = data["unified_id"]

    entries = person["entries"]

    for entry in entries:
        composite_id = entry["composite_id"]
        combo = (unified_id, composite_id)

        if combo not in seen_combinations:
            seen_combinations.add(combo)
            b2p_matched_list.append({
              "unified_id": unified_id,
              **entry,
              "person_id": person_id,
              "person_id_old": id_old
           })
        else:
            existing = [e for e in b2p_matched_list if e["person_id"] == person_id and e["composite_id"] == composite_id][0]

            existing.update({
               role: existing[role] or entry[role] for role in roles
            })
            update_count += 1

            updates[unified_id] = {
               "count": update_count,
               "entry_old": entry,
               "existing_updated": existing
            }

        b2p_matched_data[person_id] = {
           "person_id": person_id,
           "person_id_old": id_old,
           "unified_id": unified_id,
           "entries": entries
        }

prep_folder_path = Path(project_root / "data/people/prepping4loading")

b2p_matched_data_file = Path(project_root / prep_folder_path / "b2p_matched_data.json")
b2p_matched_list_file = Path(project_root / prep_folder_path / "b2p_matched_list.json")

with open(b2p_matched_data_file, "w") as f:
    json.dump(b2p_matched_data, f, ensure_ascii=False, indent=2)
with open(b2p_matched_list_file, "w") as f:
    json.dump(b2p_matched_list, f, ensure_ascii=False, indent=2)



rprint(f"""
=== LOG ===
total matches: {len(matched_dict)}
b2p_data len: {len(b2p_matched_data)}
b2p_list len: {len(b2p_matched_list)}
problems len: {len(problems)}
updated: {update_count}
len updated: {len(updates)}
=== DONE ===
""")

rprint(problems)

=== LOG ===
total matches: 5103
b2p_data len: 5103
b2p_list len: 9234
problems len: 0
updated: 130
len updated: 116
=== DONE ===

{}

In [9]:
# # unified_id is actually not as stable as the old ids


# for unified_id, person in matched_dict.items():
# # for unified_id, person in list(matched_dict.items())[:5]:
#     if not unified_id in mapping_dict:
#        problem_count += 1
#        problems[unified_id] = person
#        continue

#     data = mapping_dict[unified_id]
#     person_id = data["person_id"]
#     person_id_old = person["person_id_old"]
#     entries = person["entries"]

#     for entry in entries:
#         composite_id = entry["composite_id"]
#         combo = (unified_id, composite_id)

#         if combo not in seen_combinations:
#            seen_combinations.add(combo)
#            b2p_matched_list.append({
#               "unified_id": unified_id,
#               **entry,
#               "person_id": person_id,
#               "person_id_old": person_id_old
#            })

#         else:
#             existing = [e for e in b2p_matched_list if e["unified_id"] == unified_id and e["composite_id"] == composite_id][0]
#             existing.update({
#                role: existing[role] or entry[role] for role in roles
#             })
#             update_count += 1
#             updates[unified_id] = {
#                "count": update_count,
#                "entry_old": entry,
#                "existing_updated": existing
#             }

#         b2p_matched_data[person_id] = {
#            "person_id": person_id,
#            "person_id_old": person_id_old,
#            "unified_id": unified_id,
#            "entries": entries
#         }

# rprint(problems)
# with open("problems.json", "w") as f:
#     json.dump(problems, f, ensure_ascii=False, indent=2)

# rprint(f"""
# === LOG ===
# total matches: {len(matched_dict)}
# b2p_data len: {len(b2p_matched_data)}
# b2p_list len: {len(b2p_matched_list)}
# updated: {update_count}
# len updated: {len(updates)}
# === DONE ===
# """)
